In [2]:
from bs4 import BeautifulSoup
import requests
import re
import networkx as nx
import spacy
import time
import math
import pickle
import json
from composer import *
from ipysigma import Sigma


"""les relations d'influence entre compositeurs sont qualifiées plus précisément qu'avec la seule présence de l'un dans la page wikipedia de l'autre grâce à la fonction orienting with dates"""
"""compléter avec les influences - même peu nombreuses - spécifiées dans wikidata, en plus de celles des élèves - réfléchir jusqu'où aller dans la profondeur des élèves """
"""ajouter une détection des liens vers des pages de compositeurs sur wikipedia"""

def get_person_entities(text):
    nlp = spacy.load("en_core_web_sm")
    doc = nlp(text)
    L=[]
    for ent in doc.ents:
        if ent.label_=='PERSON':
            #print(ent.text, ent.start_char, ent.end_char, ent.label_)
            L.append(ent.text)
    #print(L)
    return L

def get_wikipedia_page_text(name,language="en"):
    url = f"https://{language}.wikipedia.org/w/index.php?search={name}"
    page = requests.get(url)
    soup = BeautifulSoup(page.content, "html.parser")
    text=soup.get_text()
    if language =="en" and (name+" - Wikipedia" in text or name.split(" ")[-1]+" - Wikipedia" in text):
        return get_wikipedia_page_text(name+" (composer)")
    elif language=="fr" and (name+" - Wikipédia" in text or name.split(" ")[-1]+" - Wikipédia" in text):
        return get_wikipedia_page_text(name+" (compositeur)")
    return text

def composer_filter_by_dict(list_of_names, dico):
    """filtre les noms qui sont ceux de compositeurs à partir d'un dictionnaire de compositeurs, évite d'avoir à rechercher dans wikidata à chaque fois donc plus rapide mais moins robuste, à compléter peut-être avec le TD qui utilisait des Trie"""
    list_of_names = list(filter(lambda x: x.strip() in dico, list_of_names))
    return list_of_names

#composer_filter_by_dict(["Richard Wagner","Bach","Mozart"],{"Richard Wagner":(1813,1883),"Johann Sebastian Bach":(1685,1750)})

def composer_filter(list_of_names):
    compo_dico={}
    cpt=0
    for n in list_of_names:
        c_id = find_id(n)
        if c_id is not None and c_id not in compo_dico:
            c = is_composer(c_id,n)
            #print(is_composer(n))
            if c is not None :
                if cpt==0:
                    compo=c
                compo_dico[c_id]=c
                print(n,c_id,f"is the composer number {cpt} found in {compo}'s page")
                cpt+=1
    return compo,compo_dico


def find_compositional_influences(name,fr=False):#problème : l'algo saute souvent Beethoven, y compris chez Wagner
    content = get_wikipedia_page_text(name)
    if fr:
        content+=get_wikipedia_page_text(name,"fr")
    persons = get_person_entities(content)
    return(composer_filter(persons))

def build_list_of_influences(enum_of_composers,french=False):
    list_of_influ_dict=[]
    for composer in enum_of_composers:
        list_of_influ_dict.append(find_compositional_influences(composer,fr=french))
    return list_of_influ_dict



def orienting_with_dates(list_of_dico,compo_dico={}):
    """completing the influences of all encountered compsoers; orienting the influence relations when no doubt exists because of birth-death dates"""

    for compo,dico in list_of_dico:
        cpt=0
        b=False #boolean equal to True iff we have access the the birth date of the first composer
        d=False #boolean equal to True iff we have access the the death date of the first composer
        #print(dico)
        for c in dico.values():

            if c.id not in compo_dico :
                compo_dico[c.id]=c

            if cpt==0: #the first element of each dictionary is the one whose wikipedia page was used to retrieve all the others present in the current dictionary
                #compo = compo_dico[c.id]
                cpt+=1
                print("1st composer of this page :",compo,compo.id,compo_dico[compo.id].influences)
                birth_date_compo = compo.birth
                death_date_compo = compo.death
                if birth_date_compo is not None :
                    print("db:"+str(compo.birth))
                    b = True

                if death_date_compo is not None :
                    print("dd:"+str(compo.death))
                    d = True

            else :
                birth_date = c.birth
                death_date = c.death

                if d and birth_date is not None:
                    if birth_date > death_date_compo-20:#if compo died at most 2O years after the birth of c then we assume only compo could influence c and not the other way around
                        compo_dico[compo.id].influences.add(c)
                        #print(f"added {c} to {compo_dico[compo.id]}")
                    elif b and death_date is not None:
                        if death_date < birth_date_compo+20:#if the current composer c died at most 20 years after the birth of compo we assume compo could not influence c and that only c influenced compo
                            compo_dico[c.id].influences.add(compo)
                            #print(f"added {compo_dico[compo.id]} to {c}")
                        else:
                            compo_dico[compo.id].influences.add(c)
                            #print(f"added {c} to {compo_dico[compo.id]} in the new way")
                            compo_dico[c.id].influences.add(compo)
                            #print(f"added {compo_dico[compo.id]} to {c}")
                    else:
                        compo_dico[compo.id].influences.add(c)
                        #print(f"added {c} to {compo_dico[compo.id]}")
                        compo_dico[c.id].influences.add(compo)
                        #print(f"added {compo_dico[compo.id]} to {c}")

                elif b and death_date is not None:
                        if death_date < birth_date_compo+20:#if the current composer c died at most 20 years after the birth of compo we assume compo could not influence c and that only c influenced compo
                            compo_dico[c.id].influences.add(compo)
                            #print(f"added {compo_dico[compo.id]} to {c}")
                        #elif d and birth_date is not None:
                        #    if birth_date > death_date_compo-20:#if compo died at most 2O years after the birth of c then we assume only compo could influence c and not the other way around
                        #        compo_dico[compo.id].influences.add(c)
                        else:
                            compo_dico[compo.id].influences.add(c)
                            #print(f"added {c} to {compo_dico[compo.id]}")
                            compo_dico[c.id].influences.add(compo)
                            #print(f"added {compo_dico[compo.id]} to {c}")

                else:#we cannot say anything for sure since they were contemporary so by default we assume they influenced each other
                    compo_dico[compo.id].influences.add(c)
                    #print(f"added {c} to {compo_dico[compo.id]}")
                    compo_dico[c.id].influences.add(compo)
                    #print(f"added {compo_dico[compo.id]} to {c}")
        print(compo_dico[compo.id],"has influenced",compo_dico[compo.id].influences)

    return compo_dico



def build_graph_from_dict(compo_dico,G=nx.DiGraph()):
    for compo in compo_dico.values():
        print("in build_graph_from_dict",compo,"has influenced",compo.influences)
        for c in compo.influences:
            G.add_edge(compo,c)
            print(f"added {c} to {compo_dico[compo.id]}")

    # save graph object to file
    pickle.dump(G, open(str(next(iter(compo_dico.keys())))+".pickle", 'wb'))
    return G

def graph_of_influences(list_of_compo_names,fr=True,G=nx.DiGraph()):
    list_of_influ_dicts=build_list_of_influences(list_of_compo_names,fr)#put false if you're only interested in the english wiki
    compo_dico = orienting_with_dates(list_of_influ_dicts)
    return build_graph_from_dict(compo_dico,G)


In [21]:
def prepare_sigma_with_composer_ids(original_graph):
    """Relabels a graph using Composer.id as node labels and returns a Sigma widget."""
    import math  # make sure math is imported
    id_map = {node: str(node)+" - "+str(node.id) for node in original_graph.nodes()}
    graph = nx.relabel_nodes(original_graph, id_map)

    colors = {
        "Medieval or Renaissance": "#654321",      # Brown
        "Baroque": "#2ECC71",       # Green
        "Classique": "#F1C40F",     # Yellow
        "Romantique": "#E74C3C",    # Red
        "XXe": "#3498DB",           # Blue
        "contemporain": "#7F00FF",  # Lilac
        "Unknown": "#95A5A6"        # Gray
    }

    """# Clean node attributes and apply era-based color
    for n, d in graph.nodes(data=True):
        for k, v in d.items():
            if not isinstance(v, (str, int, float, bool, type(None))):
                d[k] = str(v)"""
    
    for node in original_graph.nodes():
        era = node.get_era()  # get era from original node object
        #era = "Unknown"
        graph.nodes[str(node)+" - "+str(node.id)]["color_node"] = colors.get(era, "#95A5A6")  # default gray
    
    for u, v, d in graph.edges(data=True):
        original_source_node = [n for n in original_graph.nodes() if f"{n} - {n.id}" == u][0]
        source_era = original_source_node.get_era()
        d["color_edge"] = colors.get(source_era, "#95A5A6")
        for k, val in d.items():
            if not isinstance(val, (str, int, float, bool, type(None))):
                d[k] = str(val)


    # Layout
    layout = {
        node: {
            "x": math.cos(i * 2 * math.pi / len(graph)),
            "y": math.sin(i * 2 * math.pi / len(graph)),
        }
        for i, node in enumerate(graph)
    }

    sigma = Sigma(
        graph=graph,
        layout=layout,
        default_edge_type="curve",
        node_border_color_from="node",
        raw_node_color="color_node",  # use the color based on era
        raw_edge_color="color_edge",
        label_font="cursive",
        node_size=graph.out_degree()
    )

    return sigma

#prepare_sigma_with_composer_ids(G)
import pickle
from ipysigma import Sigma

def sigma_graph_from_file(filename):
    G = pickle.load(open(filename, 'rb'))
    return G,prepare_sigma_with_composer_ids(G)
    



G=pickle.load(open("Q174606.pickle", 'rb'))
#G=pickle.load(open("Q205519.pickle", 'rb'))
S = prepare_sigma_with_composer_ids(G)
S

Sigma(nx.DiGraph with 1,889 nodes and 4,888 edges)

In [26]:
from composer import * 
#g=pickle.load(open("Q964256.pickle", 'rb'))
#g=pickle.load(open("Q1339.pickle",'rb'))
g=pickle.load(open("Q174606.pickle", 'rb'))
id_map={node: str(node)+" - "+str(node.id) for node in g.nodes()}
graph = nx.relabel_nodes(g, id_map)

# Clean all node/edge attributes to make sure they're JSON serializable
for n, d in graph.nodes(data=True):
    for k, v in d.items():
        if not isinstance(v, (str, int, float, bool, type(None))):
            d[k] = str(v)

for u, v, d in graph.edges(data=True):
    for k, val in d.items():
        if not isinstance(val, (str, int, float, bool, type(None))):
            d[k] = str(val)
"""
layout = {
    node: {
        "x": math.cos(i * 2 * math.pi / len(graph)),
        "y": math.sin(i * 2 * math.pi / len(graph)),
    }
    for i, node in enumerate(graph)
}

#pagerank_scores = nx.pagerank(graph)

for k, v in pagerank_scores.items():
    assert isinstance(v, (int, float)), f"Non-numeric pagerank for node {k}: {v}"

era_categories = ["Medieval","Baroque", "Classique", "Romantique", "XXe", "contemporain"]
colors = {
    "Medieval": "#654321", #Brown
    "Baroque": "#2ECC71",      # Green
    "Classique": "#F1C40F",    # Yellow
    "Romantique": "#E74C3C",       # Red
    "XXe": "#3498DB",  # Blue
    "contemporain": "#7F00FF", #Lilac
    "Unknown": "#95A5A6"
    
}"""

#Girvan-Newman

import itertools
import matplotlib.colors as mcolors
#N=79
#comp = nx.community.girvan_newman(K)
#for communities in itertools.islice(comp, N):
#    print(len(tuple(sorted(c) for c in communities)))
k = 6 #number of communities we want to separate
comp = nx.community.girvan_newman(g)
limited = itertools.takewhile(lambda c: len(c) <= k, comp)
i=1
for communities in limited:
    print("number of communities with Girvan-Newman algorithm after", str(i), " iterations : ", len(communities))#tuple(sorted(c) for c in communities)
    i+=1
#print(tuple(sorted(c) for c in communities),"\n")
#print(communities)
# Generate a list of distinct colors
color_list = list(mcolors.TABLEAU_COLORS.values()) + list(mcolors.CSS4_COLORS.values())
np.random.shuffle(color_list)

# Assign a unique color to each community
node_to_color = {}
for i, community in enumerate(communities):
    color = color_list[i % len(color_list)]
    for node in community:
        label = f"{str(node)} - {node.id}"
        node_to_color[label] = color

# Apply colors to nodes
for node_label in graph.nodes:
    graph.nodes[node_label]["color_node"] = node_to_color.get(node_label, "#95A5A6")  # Gray fallback

partitions = {"Medieval":[],"Baroque":[], "Classique":[], "Romantique":[], "XXe":[], "contemporain":[],"Unknown":[]}

for node in g.nodes:
    era = node.get_era()  # Default to "medium"
    #graph.nodes[str(node)+" - "+str(node.id)]["color_node"] = color.get(era, "#95A5A6")  # Gray if unknown
    #g.nodes[node]["size"] = 10  # Set a default size
    #partitions[era].append(graph.nodes[str(node)+" - "+str(node.id)])
    partitions[era].append(str(node)+" - "+str(node.id))


for source, target in graph.edges:
    graph.edges[source, target]["color_edge"] = graph.nodes[target]["color_node"]

layout = {}

# Grid positions for each era cluster center
grid_positions = {
    "Medieval": (0, 0),
    "Baroque": (-3, 0),
    "Classique": (0, 2),
    "Romantique": (3, 0),
    "XXe": (-2, -2),
    "contemporain": (2, -2)
}

# Assign circular layouts within each cluster
for era, node_names in partitions.items():
    base_x, base_y = grid_positions.get(era, (0, 0))
    angle_step = 2 * np.pi / max(1, len(node_names))
    radius = 1.5  # Distance from center of cluster

    for i, node_name in enumerate(node_names):
        angle = i * angle_step
        layout[node_name] = {
            "x": base_x + radius * np.cos(angle),
            "y": base_y + radius * np.sin(angle)
        }

# Visualize with ipysigma
sigma = Sigma(
    graph,
    raw_node_color="color_node",
    raw_edge_color="color_edge",
    layout=layout,
    node_label="label",
    default_edge_type="curve",
    node_border_color_from="node",
    label_font="cursive",
    node_size=graph.out_degree()
    #nx.pagerank(graph)
)

sigma

number of communities with Girvan-Newman algorithm after 1  iterations :  2
number of communities with Girvan-Newman algorithm after 2  iterations :  3
number of communities with Girvan-Newman algorithm after 3  iterations :  4
number of communities with Girvan-Newman algorithm after 4  iterations :  5
number of communities with Girvan-Newman algorithm after 5  iterations :  6


Sigma(nx.DiGraph with 1,889 nodes and 4,888 edges)

In [32]:
import json

# Prepare export data
export_data = {
    "nodes": [
        {
            "id": str(n),
            "x": layout.get(n, {}).get("x", 0),
            "y": layout.get(n, {}).get("y", 0),
            "label": graph.nodes[n].get("label", str(n)),
            "color": graph.nodes[n].get("color_node", "#999999"),
            "size": graph.nodes[n].get("size", 1)
        }
        for n in graph.nodes
    ],
    "edges": [
        {
            "source": u,
            "target": v,
            "color": graph.edges[u, v].get("color_edge", "#999999"),
            "type": graph.edges[u, v].get("type", "curve")
        }
        for u, v in graph.edges
    ]
}

# Write to file
with open("sigma_girvan_newman_graph.json", "w") as f:
    json.dump(export_data, f, indent=2)

In [28]:
def add_chroma_edges(G_influ,G_chroma):
    for n in G_influ.nodes():
        for m in G_chroma.nodes():
            if n.id==find_id(m.split(";")):
                for neigh in G_chroma.neighbors(m):
                    #if neigh.id in map(get_id,G_influ.nodes())
                    G_influ.add_edge(n,neigh)
                    G_influ.add_edge(neigh,n)
    return G_influ

G=pickle.load(open("graphe_centroids_pot_seuil=1.3.pkl", 'rb'))
#S = prepare_sigma_with_composer_ids(G)
import math

# Définir une colormap bleu-rouge pour les poids des arêtes
import matplotlib
cmap = matplotlib.cm.get_cmap('viridis')

# Normaliser les poids pour la colormap
weights = np.array([d['weight'] for _, _, d in G.edges(data=True)])
w_min, w_max = weights.min(), weights.max()

for u, v, d in G.edges(data=True):
    norm_weight = (d['weight'] - w_min) / (w_max - w_min) if w_max > w_min else 0.5
    rgba = cmap(norm_weight)
    # Convertir en couleur hexadécimale
    d['color_edge'] = matplotlib.colors.rgb2hex(rgba)

layout = {}
grid_positions = {}

angle_step = 2 * math.pi / max(1,len(G.nodes()))
radius = 1
for i,node in enumerate(G.nodes()):
    angle = i * angle_step
    layout[node] = {
        "x": radius * math.cos(angle),
        "y": radius * math.sin(angle)
    }

# Calcul d'un score de PageRank pour ajuster la taille des noeuds par exemple
#pagerank_scores = nx.pagerank(G)

# Visualisation avec ipysigma
sigma = Sigma(
    graph=G,
    raw_node_color="color",
    raw_edge_color="color_edge",
    layout=layout,
    node_label="nom complet",
    default_edge_type="curve",
    label_font="cursive",
    node_size=G.degree()#pagerank_scores,  
)

sigma

/var/folders/6_/wjb9ddn965xckzzthm1x7rnr0000gn/T/ipykernel_8289/2446314615.py:17: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  cmap = matplotlib.cm.get_cmap('viridis')


Sigma(nx.Graph with 70 nodes and 385 edges)